<a href="https://colab.research.google.com/github/zhesun0304/ECON3916/blob/main/The_Architecture_of_Control.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Step 1: Environment and Data Strategy
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

# Loading dataset here
df = pd.read_csv('lalonde.csv')

df.head()

,Unnamed: 0,treat,age,educ,black,hispan,married,nodegree,re74,re75,re78
0,1,1,37,11,1,0,1,1,0.0,0.0,9930.0460
1,2,1,22,9,0,1,0,1,0.0,0.0,3595.8940
2,3,1,30,12,1,0,0,0,0.0,0.0,24909.4500
3,4,1,27,11,1,0,0,1,0.0,0.0,7506.1460
4,5,1,33,8,1,0,0,1,0.0,0.0,289.7899


In [2]:
# Step 2: The Observational Failure
# Naive Comparison
naive_diff = df.loc[df['treat'] == 1, 're78'].mean() - df.loc[df['treat'] == 0, 're78'].mean()
print(f"Naive Difference in Means: ${naive_diff:,.2f}")


Naive Difference in Means: $-635.03


In [3]:
# Step 3: Propensity Score Estimation
# Covariates
X = df[['age', 'educ', 'black', 'hispan', 'married', 'nodegree', 're74', 're75']]
y = df['treat']

# Fit Propensity Model
logit = LogisticRegression(solver='liblinear')
logit.fit(X, y)

# Generate Scores
df['pscore'] = logit.predict_proba(X)[:, 1]

df[['treat', 'pscore']].head()

,treat,pscore
0,1,0.443350
1,1,0.144660
2,1,0.722355
3,1,0.664151
4,1,0.698286


In [4]:
# Step 4: The Matching Algorithm (Nearest Neighbor)
from sklearn.neighbors import NearestNeighbors

# Separate groups
treated = df[df.treat==1]
control = df[df.treat==0]

# Fit NN on Control scores
nbrs = NearestNeighbors(n_neighbors=1).fit(control[['pscore']])

# Find matches for Treated scores
distances, indices = nbrs.kneighbors(treated[['pscore']])
matched_control = control.iloc[indices.flatten()].copy()

# Construct Matched DataFrame
matched_df = pd.concat([treated, matched_control], ignore_index=True)

matched_df.head()

,Unnamed: 0,treat,age,educ,black,hispan,married,nodegree,re74,re75,re78,pscore
0,1,1,37,11,1,0,1,1,0.0,0.0,9930.0460,0.443350
1,2,1,22,9,0,1,0,1,0.0,0.0,3595.8940,0.144660
2,3,1,30,12,1,0,0,0,0.0,0.0,24909.4500,0.722355
3,4,1,27,11,1,0,0,1,0.0,0.0,7506.1460,0.664151
4,5,1,33,8,1,0,0,1,0.0,0.0,289.7899,0.698286


In [5]:
# Step 5: Assessing Balance and Estimating the Effect
from scipy import stats

# T-test on raw data
diff = treated['re78'].mean() - control['re78'].mean()
t_stat, p_val = stats.ttest_ind(treated['re78'], control['re78'])

print(f"Raw Effect (Difference): ${diff:,.2f}")
print(f"P-value: {p_val:.4f}")

# Isolate the matched outcomes
matched_treated = matched_df[matched_df.treat==1]['re78']
matched_control = matched_df[matched_df.treat==0]['re78']

# Estimate the causal effect (T-test on matched data)
matched_diff = matched_treated.mean() - matched_control.mean()
t_stat, p_val = stats.ttest_ind(matched_treated, matched_control)


print(f"Recovered Effect (Matched Difference): ${matched_diff:,.2f}")
print(f"P-value: {p_val:.4f}")

Raw Effect (Difference): $-635.03
P-value: 0.3342
Recovered Effect (Matched Difference): $583.04
P-value: 0.4438
